# X_EXTRAS

This notebook generates a LaTeX table of observables used in the sweep tests. It reads `dd` from `recipe.py`, loads the sweep parameter JSON files, and writes the final table to `output/tex/observables_table.tex`.


## Inputs and logic

The notebook uses these inputs:

- `recipe.py` for the list of observable dictionaries `dd`
- `output/sweeps/gbm_params.json`
- `output/sweeps/qrf_params.json`
- `output/sweeps/sim_params.json`

Only observables present in at least one `obs_sel` list are included. If `sim_params.json` is missing, the SIM column defaults to `\boolFalse` for all rows. Labels found in `obs_sel` but missing from `dd` are skipped with a warning.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

from recipe import dd


def load_obs_sel(path: Path):
    if not path.exists():
        print(f"Warning: missing file {path}; using empty obs_sel")
        return []
    with path.open() as f:
        data = json.load(f)
    obs_sel = data.get("obs_sel", [])
    if obs_sel is None:
        return []
    if not isinstance(obs_sel, list):
        print(f"Warning: obs_sel in {path} is not a list; using empty obs_sel")
        return []
    return obs_sel


def latex_escape(text: str) -> str:
    return str(text).replace('_', r'\_')


def ref_to_latex(reference):
    if not reference:
        return ""
    if isinstance(reference, str):
        refs = [reference]
    elif isinstance(reference, list):
        refs = [str(r) for r in reference if r]
    else:
        refs = [str(reference)]
    refs = [r.strip() for r in refs if str(r).strip()]
    if not refs:
        return ""
    return r"\citet{" + ", ".join(refs) + "}"


def bool_tex(flag: bool) -> str:
    return r"\boolTrue" if flag else r"\boolFalse"


## Generate table

This cell reads the sweep parameter files, filters observables to those used in at least one sweep, preserves the order from `dd`, and writes the LaTeX output file.


In [ ]:
root = Path('.')
sweep_dir = root / 'output' / 'sweeps'
out_path = root / 'output' / 'tex' / 'observables_table.tex'

gbm_obs = set(load_obs_sel(sweep_dir / 'gbm_params.json'))
qrf_obs = set(load_obs_sel(sweep_dir / 'qrf_params.json'))
sim_obs = set(load_obs_sel(sweep_dir / 'sim_params.json'))

used = gbm_obs | qrf_obs | sim_obs

rows = []
seen = set()
for item in dd:
    label = item.get('label')
    if label in used:
        seen.add(label)
        description = item.get('description', '') or ''
        reference = ref_to_latex(item.get('reference', ''))
        rows.append(
            f"{latex_escape(label)} & {description} & {bool_tex(label in gbm_obs)} & {bool_tex(label in qrf_obs)} & {bool_tex(label in sim_obs)} & {reference} \\\"
        )

missing = sorted(used - seen)
for label in missing:
    print(f"Warning: label '{label}' present in obs_sel but missing from dd; skipping")

table = "\n".join([
    r"\begin{table}",
    r"    \centering",
    r"    \newcolumntype{C}[1]{>{\centering\arraybackslash}p{#1}}",
    "",
    r"    \begin{tabular}{p{3cm} p{3.7cm} C{0.7cm} C{0.7cm} C{0.7cm} p{3.5cm}}",
    "",
    r"    Label & Description & GBM & QRF & SIM &  Data and processing\\",
    r"    \midrule",
    *["    " + row for row in rows],
    r"    \bottomrule",
    r"    \end{tabular}",
    r"    \caption{Observables used.}",
    r"    \label{tab:observables}",
    r"\end{table}",
    "",
])

out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(table)
print(f"Wrote {out_path}")


## Preview

This final cell prints the generated LaTeX so you can inspect it directly in the notebook before using it in your manuscript.


In [ ]:
print(out_path.read_text())
